## MLflow's Model Registry

In [1]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (
/home/codespace/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


### Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI

In [10]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
client.search_experiments()

[<Experiment: artifact_location='/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/mlruns/5', creation_time=1773059541182, experiment_id='5', last_update_time=1773059541182, lifecycle_stage='active', name='nyc-taxi-gbregressor', tags={}>,
 <Experiment: artifact_location='/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/mlruns/4', creation_time=1773046235283, experiment_id='4', last_update_time=1773046235283, lifecycle_stage='active', name='my-cool-new-experiment', tags={}>,
 <Experiment: artifact_location='/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/mlruns/2', creation_time=1772635207768, experiment_id='2', last_update_time=1772635207768, lifecycle_stage='active', name='xgboost-hyperopt-tuning', tags={}>,
 <Experiment: artifact_location='/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/mlruns/1', creation_time=1772541828979, experiment_id='1', last_update_time=1772541828979, lifecycle_stage='active', name='nyc-taxi-experiment'

In [5]:
client.create_experiment(name="my-cool-new-experiment")

'4'

Let's check the latest versions for the experiment with id `5`, which is 'nyc-taxi-gbregressor'...

In [11]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='5',
    filter_string="metrics.rmse > 5.5",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [12]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 31571bbb326249e3b9cfa0314f52b657, rmse: 5.5316
run id: 7da7b4e06e1f470d91c933a017fc4306, rmse: 5.5316
run id: af83f4152a2b4023a813591d1653bc2d, rmse: 7.6377
run id: 9c1cda9d2e0f4b50af45a7646d5a7be6, rmse: 7.6680
run id: a066261ec40343b08d3b2b70b94e9c56, rmse: 7.6877


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `5` was created.
3. Transition the version `5` to "Staging" and adding annotations to it.

In [5]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [7]:
run_id = "31571bbb326249e3b9cfa0314f52b657"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

2026/03/09 12:53:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/09 12:53:40 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Successfully registered model 'nyc-taxi-regressor'.
2026/03/09 12:53:41 WARNING mlflow.tracking._model_registry.fluent: Run with id 31571bbb326249e3b9cfa0314f52b657 has no artifacts at artifact path 'model', registering model based on models:/m-3e98940cee92406fbebe42b183a46d32 instead
Created version '1' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1773060821033, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1773060821033, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='31571bbb326249e3b9cfa0314f52b657', run_link=None, source='models:/m-3e98940cee92406fbebe42b183a46d32', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [52]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 5, stage: None
version: 4, stage: Staging


/tmp/ipykernel_14688/669935608.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [19]:
model_version = 4
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_14688/1718116654.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1773062447556, current_stage='Staging', deployment_job_state=None, description='', last_updated_timestamp=1773062601662, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='', run_link='', source='/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/mlruns/5/models/m-d71c37ee2f7144fcbf3f3c5cfbe36ad4/artifacts', status='READY', status_message=None, tags={}, user_id=None, version=4>

In [20]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1773062447556, current_stage='Staging', deployment_job_state=None, description='The model version 4 was transitioned to Staging on 2026-03-09', last_updated_timestamp=1773062617066, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='', run_link='', source='/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/mlruns/5/models/m-d71c37ee2f7144fcbf3f3c5cfbe36ad4/artifacts', status='READY', status_message=None, tags={}, user_id=None, version=4>

In [67]:
model_version = 6
new_stage = "Production"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_14688/2085341095.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1773065689684, current_stage='Production', deployment_job_state=None, description='', last_updated_timestamp=1773065769362, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='', run_link='', source='/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/mlruns/5/models/m-f324e80df1d6460e9b2f05fe6982b90e/artifacts', status='READY', status_message=None, tags={}, user_id=None, version=6>

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [64]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    # Try pyfunc first, fall back to sklearn flavor
    try:
        model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    except Exception:
        model = mlflow.sklearn.load_model(f"models:/{name}/{stage}")
    
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [35]:
df = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-03.parquet')

In [68]:
client.download_artifacts(run_id='f95e2aacf16042898e22feccea7d314b', path='preprocessor', dst_path='.')

'/workspaces/TravelTimeOptimisation_MLOps/02-ExperimentTracking/preprocessor'

In [69]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [70]:
X_test = preprocess(df, dv)

In [71]:
target = "duration"
y_test = df[target].values

In [72]:
X_test

<48336x4162 sparse matrix of type '<class 'numpy.float64'>'
	with 93697 stored elements in Compressed Sparse Row format>

In [ ]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

In [ ]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)